# Conformal Admission Control on Real Azure Traces\n\nThis notebook demonstrates an **ACI (Adaptive Conformal Inference) admission controller** — an online policy that decides, request-by-request, whether to admit each incoming request to an overloaded queue, while keeping the realized violation (SLO-breach) rate near a target `alpha`.\n\nThe controller is compared against four baselines on real [Azure Functions 2019](https://github.com/Azure/AzurePublicDataset) trace-derived data, across five traffic regimes (`stationary`, `burst`, `drift`, `regime_switch`, `adversarial`):\n\n- **ConformalPolicy** — the ACI policy: `lambda_{t+1} = lambda_t + eta * (alpha - y_t)`, admits iff `risk_score <= lambda_t`.\n- **FixedThresholdPolicy** — a threshold tuned once on warm-up data, then frozen forever (\"no adaptation\" baseline).\n- **MisspecifiedIndexPolicy** — a deliberately misspecified M/M/1-style queueing-theory baseline, fit only on the stationary regime.\n- **FrozenRLPolicy** — a closed-form logistic-regression contextual-bandit substitute, trained once and frozen.\n- **OracleHindsightPolicy** — a hindsight-optimal upper bound that knows the true labels in advance.\n\nData loading is kept strictly separate from policy code, and policies only ever see ground-truth outcome labels through an explicit `update()` feedback call inside the replay loop — this mirrors the original `method.py` structure exactly, just split into notebook cells with explanatory markdown between sections.\n\n**This is a small-scale demo** (curated subset of the full 210,000-row dataset, reduced warm-up/window/bootstrap sizes) so it runs quickly end-to-end. The original code's logic, formulas, and structure are otherwise unchanged.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru -- NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')

# numpy, matplotlib -- pre-installed on Colab, install locally only (to match Colab's exact env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations

import gc
import json
import sys
import time
from collections import defaultdict
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

REGIMES = ["stationary", "burst", "drift", "regime_switch", "adversarial"]
POLICIES = ["conformal", "fixed_threshold", "misspecified_index", "frozen_rl", "oracle"]
DOCUMENTED_VIOLATION_RATES = {
    "stationary": 0.0395,
    "burst": 0.0024,
    "drift": 0.1553,
    "regime_switch": 0.0309,
    "adversarial": 0.3825,
}

## Load the demo data\n\n`mini_demo_data.json` is a curated, regime-stratified, chronologically-sorted subset (240 rows per regime, 1200 rows total) of the full 210,000-row Azure-trace-derived dataset. We try the GitHub raw URL first (so this notebook also works standalone on Colab), falling back to the local file.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-db8806-conformal-admission-control-distribution/main/round-2/experiment-1/demo/mini_demo_data.json"
import os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
raw_examples = data["datasets"][0]["examples"]
print(f"Loaded {len(raw_examples)} raw rows")
print(raw_examples[0])

## Config\n\nAll tunable parameters live here. Values below are reduced from the original script (`ALPHA`, `ETAS` unchanged -- those are the pre-registered scientific settings) so the demo runs in seconds on the small curated dataset instead of ~35s on the full 210,000-row grid. The original values are commented alongside each one; bump these back up (and swap in `full_method_out.json`-scale data) to reproduce the full run.\n\n`VALIDATION_TOLERANCE_PP` is a new knob (the original hardcoded `1.0`): because the curated demo subset only keeps 240/720 rows per regime, its observed violation rates drift a bit further from the documented full-dataset figures than the strict 1pp tolerance allows.

In [ ]:
ALPHA = 0.10                    # target violation rate (unchanged from original)
ETAS = [0.01, 0.02, 0.05, 0.10, 0.20]  # conformal step-size sweep (unchanged, pre-registered)

WARMUP_N = 40                    # original: 200 -- rows excluded from eval per regime for threshold init
N_BOOTSTRAP = 200                # original: 10000 -- bootstrap resamples for seed-level CIs
ROLLING_WINDOW = 30               # original: 2000 -- rolling-mean window for the MAD-vs-alpha statistic
N_SEEDS = 2                       # original: 5 -- independent seeds per (policy, regime, eta) cell
N_PERM = 100                      # original: 5000 -- permutation-test resamples for Holm-corrected tests
VALIDATION_TOLERANCE_PP = 5.0     # original: 1.0 (hardcoded) -- widened for the small curated demo subset

print(f"ALPHA={ALPHA} ETAS={ETAS} WARMUP_N={WARMUP_N} N_BOOTSTRAP={N_BOOTSTRAP} "
      f"ROLLING_WINDOW={ROLLING_WINDOW} N_SEEDS={N_SEEDS} N_PERM={N_PERM}")

## MODULE 1: data_loader\n\nReads the dataset only; never imports policy code. `load_dataset` parses each raw example's JSON-encoded `input` into a flat feature dict (this is identical to the original, except it iterates the already-loaded `raw_examples` list instead of streaming `full_data_out_*.json` part files from disk). `group_by_regime_sorted` and `validate_dataset` are copied unchanged in logic, with the validation tolerance now a config variable.

In [ ]:
def load_dataset(examples: list[dict]) -> list[dict]:
    """Parse each row dict, retain only lightweight row dicts (memory-safe)."""
    parsed: list[dict] = []
    for ex in examples:
        feat = json.loads(ex["input"])
        parsed.append(
            {
                "arrival_time": float(feat["arrival_time"]),
                "risk_score": float(feat["risk_score"]),
                "slo_target": float(feat["slo_target"]),
                "regime": feat["regime_label"],
                "function_id": feat["function_id"],
                "is_synthetic": bool(feat["is_synthetic"]),
                "y": int(ex["output"]),
                # No explicit per-row `value` field exists in this dataset
                # (confirmed via preview/mini inspection) -> documented
                # fallback: uniform value=1.0. Flagged as a known
                # limitation for the paper.
                "value": 1.0,
                "realized_service_time": float(ex["metadata_service_time"]),
            }
        )
    logger.info(f"Loaded {len(parsed)} total rows")
    return parsed


def group_by_regime_sorted(parsed_rows: list[dict]) -> dict[str, list[dict]]:
    by_regime: dict[str, list[dict]] = defaultdict(list)
    for r in parsed_rows:
        by_regime[r["regime"]].append(r)
    for regime in by_regime:
        by_regime[regime].sort(key=lambda r: r["arrival_time"])
    return dict(by_regime)


def validate_dataset(by_regime: dict[str, list[dict]], n_rows_expected: int | None) -> dict:
    """Hard-fail-loud validation: schema keys and per-regime violation rates
    must match documented figures within VALIDATION_TOLERANCE_PP, else the
    loader has silently misread the schema and every downstream number
    would be wrong."""
    observed_keys = set(by_regime.keys())
    expected_keys = set(REGIMES)
    if observed_keys != expected_keys:
        raise ValueError(f"Regime keys mismatch: got {observed_keys}, expected {expected_keys}")

    report = {}
    for regime in REGIMES:
        rows = by_regime[regime]
        rate = float(np.mean([r["y"] for r in rows]))
        doc_rate = DOCUMENTED_VIOLATION_RATES[regime]
        diff_pp = abs(rate - doc_rate) * 100
        report[regime] = {
            "n_rows": len(rows),
            "observed_violation_rate": rate,
            "documented_violation_rate": doc_rate,
            "abs_diff_pp": diff_pp,
        }
        if diff_pp > VALIDATION_TOLERANCE_PP:
            raise ValueError(
                f"Regime '{regime}' violation rate {rate:.4f} deviates {diff_pp:.2f}pp "
                f"from documented {doc_rate:.4f} (>{VALIDATION_TOLERANCE_PP}pp tolerance) -- loader likely misreads schema"
            )
        logger.info(
            f"[validate] {regime}: n={len(rows)} observed={rate:.4f} documented={doc_rate:.4f} "
            f"diff={diff_pp:.3f}pp OK"
        )
    total_n = sum(len(v) for v in by_regime.values())
    if n_rows_expected is not None and total_n != n_rows_expected:
        raise ValueError(f"Total row count {total_n} != expected {n_rows_expected}")
    return report


t0 = time.time()
rows = load_dataset(raw_examples)
by_regime = group_by_regime_sorted(rows)
del rows
gc.collect()
validation_report = validate_dataset(by_regime, n_rows_expected=None)
validation_report

## MODULE 2: policy\n\nPure functions of a stream of admission-time features and externally supplied outcome labels; policies never touch ground truth `y` except through the explicit `update()` feedback call in the replay loop. All five policy classes below are copied unchanged from `method.py`.

In [ ]:
class ConformalPolicy:
    """ACI admission rule (Gibbs & Candes 2021 online gradient update),
    repurposed from prediction-interval coverage to admission control:

        lambda_{t+1} = lambda_t + eta * (alpha - y_t)   (only if request t admitted)
        admit request t  iff  risk_score(x_t) <= lambda_t

    alpha = target violation rate. eta = step size. A rejected request
    contributes no observed outcome, so lambda_t is carried forward
    unchanged for it -- this is a deliberate deviation from Gibbs & Candes'
    original setting, which always observes an outcome, and is documented
    here explicitly.
    """

    def __init__(self, alpha: float, eta: float, lambda_0: float):
        self.alpha = alpha
        self.eta = eta
        self.lam = lambda_0

    def decide(self, s_x: float, tie_break_rng: np.random.Generator | None = None) -> bool:
        return s_x <= self.lam

    def update(self, admitted: bool, y_t: int) -> None:
        if admitted:
            self.lam = self.lam + self.eta * (self.alpha - y_t)


class FixedThresholdPolicy:
    """Threshold tuned once on the stationary-regime warm-up prefix to hit
    the target alpha (via empirical quantile of risk_score at the observed
    violation rate), then FROZEN for the rest of that regime and reused
    unchanged on every other regime -- the "no adaptation" baseline."""

    def __init__(self, alpha: float, fit_rows: list[dict]):
        self.alpha = alpha
        scores = np.array([r["risk_score"] for r in fit_rows])
        ys = np.array([r["y"] for r in fit_rows])
        # threshold = risk_score quantile such that admitting scores below it
        # would have kept the empirical violation rate near alpha on warm-up
        order = np.argsort(scores)
        sorted_scores, sorted_ys = scores[order], ys[order]
        cum_violation_rate = np.cumsum(sorted_ys) / (np.arange(len(sorted_ys)) + 1)
        eligible = np.where(cum_violation_rate <= alpha)[0]
        self.lam = float(sorted_scores[eligible[-1]]) if len(eligible) else float(sorted_scores[0])

    def decide(self, s_x: float, tie_break_rng: np.random.Generator | None = None) -> bool:
        return s_x <= self.lam

    def update(self, admitted: bool, y_t: int) -> None:
        pass  # frozen: no adaptation


class MisspecifiedIndexPolicy:
    """Model-based baseline: fit a simple M/M/1-style queueing model
    (arrival rate, service-time proxy from risk_score) on the stationary
    warm-up prefix, and derive an admission threshold from its steady-state
    overflow-probability formula. Deliberately misspecified: the model
    assumptions (stationary Poisson arrivals) are wrong by construction for
    burst/drift/regime_switch/adversarial regimes, since it is fit ONLY on
    the stationary prefix and never updated."""

    def __init__(self, alpha: float, fit_rows: list[dict]):
        self.alpha = alpha
        arrivals = np.array([r["arrival_time"] for r in fit_rows])
        scores = np.array([r["risk_score"] for r in fit_rows])
        inter_arrival = np.diff(np.sort(arrivals))
        inter_arrival = inter_arrival[inter_arrival > 0]
        arrival_rate = 1.0 / np.mean(inter_arrival) if len(inter_arrival) else 1.0
        # risk_score used as a proxy "load" signal; service rate mu derived
        # from mean risk_score so that rho = lambda/mu matches observed load
        mean_score = float(np.mean(scores)) if len(scores) else 0.5
        service_rate = arrival_rate / max(mean_score, 1e-6)
        self.rho_target = self._solve_rho_for_alpha(alpha)
        # admission threshold on risk_score: admit iff score <= rho_target
        # (an M/M/1 utilization rho directly indexes overflow probability
        # rho^n; we map the target overflow prob back to an implied rho, and
        # treat risk_score as already normalized to [0,1] load units)
        self.lam = self.rho_target
        self._arrival_rate = arrival_rate
        self._service_rate = service_rate

    @staticmethod
    def _solve_rho_for_alpha(alpha: float, n_queue: int = 5) -> float:
        # M/M/1/K-style overflow probability P(overflow) ~ rho^n_queue for
        # rho<1; solve rho = alpha^(1/n_queue) as the misspecified closed-form
        return float(np.clip(alpha ** (1.0 / n_queue), 0.05, 0.95))

    def decide(self, s_x: float, tie_break_rng: np.random.Generator | None = None) -> bool:
        return s_x <= self.lam

    def update(self, admitted: bool, y_t: int) -> None:
        pass  # frozen: model never re-fit


class FrozenRLPolicy:
    """Simplified RL-style baseline (logistic-regression-on-risk_score
    contextual-bandit substitute, per the fallback plan: tabular Q-learning
    on ~2000 sparse admission rows is unstable). Trained ONCE via a closed-
    form logistic fit (Newton-Raphson / IRLS, no external deps) on the
    stationary warm-up prefix, then FROZEN (no further learning) for
    evaluation on all 5 regimes."""

    def __init__(self, alpha: float, fit_rows: list[dict], seed: int):
        self.alpha = alpha
        x = np.array([r["risk_score"] for r in fit_rows])
        y = np.array([r["y"] for r in fit_rows], dtype=float)
        self.w, self.b = self._fit_logistic(x, y, seed)
        # choose decision threshold on predicted P(violation) so that
        # admitting all rows with predicted risk <= threshold matches the
        # target alpha on the warm-up set
        p_hat = self._predict_proba(x)
        order = np.argsort(p_hat)
        sorted_p, sorted_y = p_hat[order], y[order]
        cum_rate = np.cumsum(sorted_y) / (np.arange(len(sorted_y)) + 1)
        eligible = np.where(cum_rate <= alpha)[0]
        self.p_threshold = float(sorted_p[eligible[-1]]) if len(eligible) else float(sorted_p[0])

    @staticmethod
    def _fit_logistic(x: np.ndarray, y: np.ndarray, seed: int, n_iter: int = 50) -> tuple[float, float]:
        rng = np.random.default_rng(seed)
        w, b = rng.normal(0, 0.01), 0.0
        n = len(x)
        if n == 0:
            return 0.0, 0.0
        lr = 0.5
        for _ in range(n_iter):
            z = w * x + b
            p = 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
            grad_w = np.mean((p - y) * x)
            grad_b = np.mean(p - y)
            w -= lr * grad_w
            b -= lr * grad_b
        return float(w), float(b)

    def _predict_proba(self, x: np.ndarray | float) -> np.ndarray:
        z = self.w * np.asarray(x) + self.b
        return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

    def decide(self, s_x: float, tie_break_rng: np.random.Generator | None = None) -> bool:
        p = float(self._predict_proba(s_x))
        return p <= self.p_threshold

    def update(self, admitted: bool, y_t: int) -> None:
        pass  # frozen: no further learning


class OracleHindsightPolicy:
    """Given full knowledge of this regime's y-labels in advance (evaluation
    rows only, no look-ahead beyond the regime being scored), solve the
    offline admission problem: admit the max-value subset whose realized
    violation rate <= alpha, via a greedy value/violation-cost trade-off
    (equivalent to the LP-relaxation greedy for this 0/1-value, 0/1-cost
    special case). NOT a deployable policy -- upper bound on value at
    matched safety."""

    def __init__(self, alpha: float, eval_rows: list[dict]):
        self.alpha = alpha
        n = len(eval_rows)
        budget = int(np.floor(alpha * n))
        # all rows have equal value=1.0 (documented fallback), so the optimal
        # admission set simply admits everything except enough violators to
        # respect the violation budget; ties broken by original order.
        violators_idx = [i for i, r in enumerate(eval_rows) if r["y"] == 1]
        non_violators_idx = [i for i, r in enumerate(eval_rows) if r["y"] == 0]
        keep_violators = set(violators_idx[:budget])
        self.admit_set = set(non_violators_idx) | keep_violators

    def decide_by_index(self, idx: int) -> bool:
        return idx in self.admit_set

    def decide(self, s_x: float, tie_break_rng: np.random.Generator | None = None) -> bool:
        raise RuntimeError("OracleHindsightPolicy must be driven via decide_by_index")

    def update(self, admitted: bool, y_t: int) -> None:
        pass

## MODULE 3: replay\n\nThe event loop, using both modules only through their public API. `replay_regime` steps through a regime's evaluation rows in chronological order, asks the policy to `decide`, and feeds the outcome back through `update()` -- the ONLY place ground truth reaches policy state. `rolling_mean` and `compute_metrics` compute the headline safety statistic: mean absolute deviation of the rolling admitted-request violation rate from `alpha`.

In [ ]:
def replay_regime(rows: list[dict], policy: Any, rng_seed: int) -> list[dict]:
    rng = np.random.default_rng(rng_seed)
    log = []
    is_oracle = isinstance(policy, OracleHindsightPolicy)
    for t, row in enumerate(rows):
        if is_oracle:
            admit = policy.decide_by_index(t)
        else:
            admit = policy.decide(row["risk_score"], tie_break_rng=rng)
        outcome = row["y"] if admit else None
        policy.update(admit, row["y"] if admit else 0)
        log.append(
            {
                "t": t,
                "timestamp": row["arrival_time"],
                "admit": bool(admit),
                "outcome": outcome,
                "threshold": getattr(policy, "lam", None),
                "value_if_admitted": row["value"] if admit else 0.0,
            }
        )
    return log


def rolling_mean(values: list[float], window: int) -> list[float]:
    if not values:
        return []
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    out = np.empty(n)
    csum = np.cumsum(arr)
    for i in range(n):
        lo = max(0, i - window + 1)
        s = csum[i] - (csum[lo - 1] if lo > 0 else 0.0)
        out[i] = s / (i - lo + 1)
    return out.tolist()


def compute_metrics(log: list[dict], alpha: float, window: int = ROLLING_WINDOW) -> dict:
    admitted = [e for e in log if e["admit"]]
    y = [e["outcome"] for e in admitted]
    rolling = rolling_mean(y, window)
    mad_vs_alpha = float(np.mean(np.abs(np.array(rolling) - alpha))) if rolling else float("nan")
    overall_violation_rate = float(np.mean(y)) if y else float("nan")
    total_value = float(sum(e["value_if_admitted"] for e in log))
    admit_rate = len(admitted) / len(log) if log else 0.0
    # Downsample rolling curve for storage (headline stat + a small curve)
    curve_stride = max(1, len(rolling) // 50)
    rolling_curve_sample = rolling[::curve_stride]
    return {
        "mad_vs_alpha": mad_vs_alpha,
        "overall_violation_rate": overall_violation_rate,
        "total_value": total_value,
        "admit_rate": admit_rate,
        "n_admitted": len(admitted),
        "n_total": len(log),
        "rolling_violation_rate_sample": rolling_curve_sample,
    }

## Driver helpers\n\n`build_policy` dispatches on policy name; `build_cells` enumerates every (regime, policy, eta, seed) replay cell (conformal gets the full `ETAS` sweep, other policies get a single `eta=None` cell); `_run_cell` builds a fresh policy and replays one cell. In the original, `main()` runs cells via a `ProcessPoolExecutor` once there are more than 8 cells; here we run sequentially, which is simpler in a notebook and plenty fast at demo scale.

In [ ]:
def build_policy(
    policy_name: str,
    alpha: float,
    eta: float | None,
    warmup_rows: list[dict],
    eval_rows: list[dict],
    seed: int,
    fit_rows: list[dict],
) -> Any:
    if policy_name == "conformal":
        lambda_0 = float(np.percentile([r["risk_score"] for r in warmup_rows], 90))
        return ConformalPolicy(alpha=alpha, eta=eta, lambda_0=lambda_0)
    if policy_name == "fixed_threshold":
        return FixedThresholdPolicy(alpha=alpha, fit_rows=fit_rows)
    if policy_name == "misspecified_index":
        return MisspecifiedIndexPolicy(alpha=alpha, fit_rows=fit_rows)
    if policy_name == "frozen_rl":
        return FrozenRLPolicy(alpha=alpha, fit_rows=fit_rows, seed=seed)
    if policy_name == "oracle":
        return OracleHindsightPolicy(alpha=alpha, eval_rows=eval_rows)
    raise ValueError(f"Unknown policy {policy_name}")


def _run_cell(args: tuple) -> dict:
    (regime, policy_name, eta, seed, warmup_rows, eval_rows, stationary_fit_rows) = args
    fit_rows = stationary_fit_rows if policy_name in ("frozen_rl", "misspecified_index") else warmup_rows
    policy = build_policy(
        policy_name=policy_name,
        alpha=ALPHA,
        eta=eta,
        warmup_rows=warmup_rows,
        eval_rows=eval_rows,
        seed=seed,
        fit_rows=fit_rows,
    )
    log = replay_regime(eval_rows, policy, rng_seed=seed)
    metrics = compute_metrics(log, ALPHA)
    return {"regime": regime, "policy": policy_name, "eta": eta, "seed": seed, **metrics}


def build_cells(by_regime: dict[str, list[dict]]) -> list[tuple]:
    cells = []
    stationary_fit_rows = by_regime["stationary"][:2000]
    for regime in REGIMES:
        regime_rows = by_regime[regime]
        warmup, eval_rows = regime_rows[:WARMUP_N], regime_rows[WARMUP_N:]
        for policy_name in POLICIES:
            eta_grid = ETAS if policy_name == "conformal" else [None]
            for eta in eta_grid:
                for seed in range(N_SEEDS):
                    cells.append((regime, policy_name, eta, seed, warmup, eval_rows, stationary_fit_rows))
    return cells


cells = build_cells(by_regime)
logger.info(f"Built {len(cells)} replay cells across {len(REGIMES)} regimes x {len(POLICIES)} policies")

t1 = time.time()
results = [_run_cell(c) for c in cells]
logger.info(f"Ran {len(results)} cells in {time.time() - t1:.1f}s")

## Statistics\n\n`bootstrap_ci` gives seed-level percentile-bootstrap confidence intervals; `aggregate_over_seeds` applies it per (regime, policy, eta) cell; `holm_corrected_tests` runs a permutation test (conformal at its best eta vs. each baseline, per regime) with Holm-Bonferroni correction across all comparisons; `run_knapsack_vs_fcfs` (Phase 3) compares a value-aware knapsack admission layer against plain FCFS within the same conformal eligibility set.

In [ ]:
def bootstrap_ci(values: list[float], n_boot: int = N_BOOTSTRAP, seed: int = 0) -> dict:
    arr = np.asarray([v for v in values if np.isfinite(v)], dtype=float)
    if len(arr) == 0:
        return {"mean": float("nan"), "ci_lo": float("nan"), "ci_hi": float("nan"), "n": 0}
    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot)
    n = len(arr)
    for i in range(n_boot):
        sample = arr[rng.integers(0, n, size=n)]
        boot_means[i] = sample.mean()
    return {
        "mean": float(arr.mean()),
        "ci_lo": float(np.percentile(boot_means, 2.5)),
        "ci_hi": float(np.percentile(boot_means, 97.5)),
        "n": n,
    }


def aggregate_over_seeds(results: list[dict]) -> dict:
    grouped = defaultdict(list)
    for r in results:
        key = (r["regime"], r["policy"], r["eta"])
        grouped[key].append(r)
    agg = {}
    for key, rs in grouped.items():
        agg[key] = {
            "mad_vs_alpha": bootstrap_ci([r["mad_vs_alpha"] for r in rs]),
            "overall_violation_rate": bootstrap_ci([r["overall_violation_rate"] for r in rs]),
            "total_value": bootstrap_ci([r["total_value"] for r in rs]),
            "admit_rate": bootstrap_ci([r["admit_rate"] for r in rs]),
            "n_seeds": len(rs),
        }
    return agg


def holm_corrected_tests(results: list[dict], metric: str = "mad_vs_alpha") -> list[dict]:
    """For each regime, compare conformal (best eta by mean metric) against
    each baseline via a two-sample permutation test on seed-level values,
    then apply Holm correction across all comparisons."""
    rng = np.random.default_rng(0)
    grouped = defaultdict(list)
    for r in results:
        grouped[(r["regime"], r["policy"], r["eta"])].append(r[metric])

    raw_tests = []
    for regime in REGIMES:
        # pick conformal's best eta by mean metric (lower MAD is better)
        conformal_keys = [(regime, "conformal", e) for e in ETAS]
        best_eta, best_mean = None, float("inf")
        for k in conformal_keys:
            vals = [v for v in grouped.get(k, []) if np.isfinite(v)]
            if vals and np.mean(vals) < best_mean:
                best_mean, best_eta = np.mean(vals), k[2]
        if best_eta is None:
            continue
        conformal_vals = np.array(grouped[(regime, "conformal", best_eta)])
        for baseline in ["fixed_threshold", "misspecified_index", "frozen_rl", "oracle"]:
            baseline_vals = np.array(grouped.get((regime, baseline, None), []))
            if len(baseline_vals) == 0 or len(conformal_vals) == 0:
                continue
            observed_diff = float(np.mean(conformal_vals) - np.mean(baseline_vals))
            pooled = np.concatenate([conformal_vals, baseline_vals])
            n1 = len(conformal_vals)
            n_perm = N_PERM
            count = 0
            for _ in range(n_perm):
                perm = rng.permutation(pooled)
                diff = perm[:n1].mean() - perm[n1:].mean()
                if abs(diff) >= abs(observed_diff):
                    count += 1
            p_value = (count + 1) / (n_perm + 1)
            raw_tests.append(
                {
                    "regime": regime,
                    "conformal_best_eta": best_eta,
                    "baseline": baseline,
                    "observed_diff_mad": observed_diff,
                    "p_raw": p_value,
                }
            )

    # Holm-Bonferroni correction
    m = len(raw_tests)
    order = sorted(range(m), key=lambda i: raw_tests[i]["p_raw"])
    for rank, idx in enumerate(order):
        adj = min(1.0, (m - rank) * raw_tests[idx]["p_raw"])
        raw_tests[idx]["p_holm"] = adj
    # enforce monotonicity of holm-adjusted p-values
    sorted_by_rank = [raw_tests[i] for i in order]
    running_max = 0.0
    for entry in sorted_by_rank:
        running_max = max(running_max, entry["p_holm"])
        entry["p_holm"] = running_max
        entry["significant_at_0.05"] = entry["p_holm"] < 0.05
    return raw_tests


def run_knapsack_vs_fcfs(
    by_regime: dict[str, list[dict]], best_eta_per_regime: dict[str, float], alpha: float, n_seeds: int = N_SEEDS
) -> list[dict]:
    """Phase 3: value-aware knapsack layer vs FCFS-among-eligible, using the
    same conformal eligibility set (rows with risk_score <= final lambda from
    a conformal run), comparing greedy-by-value/violation-cost admission vs
    plain first-come-first-served admission within that eligible set."""
    out = []
    for regime in REGIMES:
        regime_rows = by_regime[regime]
        warmup, eval_rows = regime_rows[:WARMUP_N], regime_rows[WARMUP_N:]
        eta = best_eta_per_regime.get(regime, 0.05)
        for seed in range(n_seeds):
            lambda_0 = float(np.percentile([r["risk_score"] for r in warmup], 90))
            policy = ConformalPolicy(alpha=alpha, eta=eta, lambda_0=lambda_0)
            log = replay_regime(eval_rows, policy, rng_seed=seed)
            eligible_idx = [e["t"] for e in log if e["admit"]]
            eligible_rows = [eval_rows[i] for i in eligible_idx]
            n_elig = len(eligible_rows)
            budget = int(np.floor(alpha * n_elig)) if n_elig else 0

            # FCFS: admit in arrival order until violation budget exhausted,
            # counting only realized violations among admitted requests
            fcfs_violations = 0
            fcfs_admitted = 0
            fcfs_value = 0.0
            for r in eligible_rows:
                if r["y"] == 1 and fcfs_violations >= budget:
                    continue
                fcfs_admitted += 1
                fcfs_value += r["value"]
                if r["y"] == 1:
                    fcfs_violations += 1
            fcfs_rate = fcfs_violations / fcfs_admitted if fcfs_admitted else float("nan")

            # Knapsack (equal-value special case -> greedy: keep all
            # non-violators, fill remaining budget with violators)
            non_v = [r for r in eligible_rows if r["y"] == 0]
            viol = [r for r in eligible_rows if r["y"] == 1]
            knap_admitted = len(non_v) + min(len(viol), budget)
            knap_value = (len(non_v) + min(len(viol), budget)) * 1.0
            knap_violations = min(len(viol), budget)
            knap_rate = knap_violations / knap_admitted if knap_admitted else float("nan")

            out.append(
                {
                    "regime": regime,
                    "seed": seed,
                    "eta_used": eta,
                    "n_eligible": n_elig,
                    "fcfs_admitted": fcfs_admitted,
                    "fcfs_value": fcfs_value,
                    "fcfs_violation_rate": fcfs_rate,
                    "knapsack_admitted": knap_admitted,
                    "knapsack_value": knap_value,
                    "knapsack_violation_rate": knap_rate,
                    "value_gain_knapsack_over_fcfs": knap_value - fcfs_value,
                }
            )
    return out

## Assemble results\n\nAggregate over seeds, sweep eta sensitivity, run the Holm-corrected pairwise tests, pick each regime's best eta, then run the knapsack-vs-FCFS phase. This mirrors the body of the original `main()` (minus the argparse / process-pool / output-file plumbing).

In [ ]:
per_cell_agg = aggregate_over_seeds(results)

eta_sensitivity_sweep = {
    regime: {
        str(eta): per_cell_agg.get((regime, "conformal", eta), None) for eta in ETAS
    }
    for regime in REGIMES
}

pairwise = holm_corrected_tests(results, metric="mad_vs_alpha")

# best eta per regime (lowest MAD, mean over seeds) for the knapsack phase
best_eta_per_regime = {}
for regime in REGIMES:
    best_eta, best_mad = ETAS[0], float("inf")
    for eta in ETAS:
        agg = per_cell_agg.get((regime, "conformal", eta))
        if agg and agg["mad_vs_alpha"]["mean"] < best_mad:
            best_mad = agg["mad_vs_alpha"]["mean"]
            best_eta = eta
    best_eta_per_regime[regime] = best_eta

knapsack_results = run_knapsack_vs_fcfs(by_regime, best_eta_per_regime, ALPHA, n_seeds=N_SEEDS)

print("best_eta_per_regime:", best_eta_per_regime)
print(f"n pairwise tests: {len(pairwise)}, n knapsack cells: {len(knapsack_results)}")
print(f"Total wall-clock so far: {time.time() - t0:.2f}s")

## Results\n\nA readable summary table (mean MAD-vs-alpha and violation rate per policy per regime, using conformal's best eta) plus two plots: the eta-sensitivity sweep for the conformal policy, and a per-regime policy comparison bar chart of the headline safety statistic.

In [ ]:
# --- summary table ---
print(f"{'regime':<15}{'policy':<20}{'eta':<8}{'mad_vs_alpha':<15}{'violation_rate':<16}{'admit_rate':<12}")
print("-" * 86)
for regime in REGIMES:
    for policy in POLICIES:
        eta = best_eta_per_regime[regime] if policy == "conformal" else None
        agg = per_cell_agg.get((regime, policy, eta))
        if agg is None:
            continue
        eta_str = f"{eta:.2f}" if eta is not None else "-"
        print(
            f"{regime:<15}{policy:<20}{eta_str:<8}"
            f"{agg['mad_vs_alpha']['mean']:<15.4f}{agg['overall_violation_rate']['mean']:<16.4f}"
            f"{agg['admit_rate']['mean']:<12.4f}"
        )

# --- plot 1: eta sensitivity sweep (conformal MAD-vs-alpha per regime) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
for regime in REGIMES:
    means = [eta_sensitivity_sweep[regime][str(eta)]["mad_vs_alpha"]["mean"] for eta in ETAS]
    ax.plot(ETAS, means, marker="o", label=regime)
ax.set_xlabel("eta (step size)")
ax.set_ylabel("MAD(rolling violation rate, alpha)")
ax.set_title("Conformal policy: eta sensitivity")
ax.legend(fontsize=8)

# --- plot 2: per-regime policy comparison (headline safety statistic) ---
ax = axes[1]
x = np.arange(len(REGIMES))
width = 0.15
for i, policy in enumerate(POLICIES):
    vals = []
    for regime in REGIMES:
        eta = best_eta_per_regime[regime] if policy == "conformal" else None
        agg = per_cell_agg.get((regime, policy, eta))
        vals.append(agg["mad_vs_alpha"]["mean"] if agg else np.nan)
    ax.bar(x + (i - 2) * width, vals, width, label=policy)
ax.set_xticks(x)
ax.set_xticklabels(REGIMES, rotation=20)
ax.set_ylabel("MAD(rolling violation rate, alpha)")
ax.set_title("Policy comparison per regime (lower = better)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

# --- knapsack vs FCFS summary ---
print("\nKnapsack vs FCFS mean value gain per regime:")
for regime in REGIMES:
    gains = [k["value_gain_knapsack_over_fcfs"] for k in knapsack_results if k["regime"] == regime]
    print(f"  {regime:<15} mean_value_gain={np.mean(gains):.3f}  (expected ~0: uniform value=1.0 fallback)")

print("\nHolm-corrected pairwise tests (conformal best-eta vs each baseline):")
for t in pairwise:
    print(
        f"  {t['regime']:<15} vs {t['baseline']:<20} diff={t['observed_diff_mad']:+.4f} "
        f"p_holm={t['p_holm']:.4f} significant={t['significant_at_0.05']}"
    )